# Enformer - regulatory effect prediction
## Description
We use **Enformer** (DeepMind, 2021 - AlphaGenome's predecessor) to predict the regulatory impact of the 7 Tier 1 variants.

In [ ]:
# Install dependencies
!pip install enformer-pytorch --quiet
!pip install einops --quiet

import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU - switch the runtime to a T4 GPU before continuing')

In [ ]:
# Imports
import torch
import numpy as np
import pandas as pd
import requests
import json
import time
import io
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LENGTH = 393216  # TF Hub Enformer requires 393,216 bp (not 196,608)
print(f'Using: {DEVICE}')

In [ ]:
# Tier 1 variants (hardcoded from the GP2 R12 analysis)
VARIANTS_T1 = [
    {'chrom': '12', 'pos': 124438876, 'ref': 'A',   'alt': 'G',
     'OR': 1.784, 'P': 8.00e-7,  'HAR': 'HAQER1191', 'sig': '↑ risk',
     'rsID': 'rs111205143',  'vep': 'enhancer/NCOR2'},
    {'chrom': '12', 'pos': 124439327, 'ref': 'GGA', 'alt': 'G',
     'OR': 0.706, 'P': 2.81e-7,  'HAR': 'HAQER1191', 'sig': '↓ protective',
     'rsID': 'rs151121336',  'vep': 'enhancer/NCOR2'},
    {'chrom': '2',  'pos': 207041,    'ref': 'G',   'alt': 'A',
     'OR': 0.681, 'P': 2.98e-11, 'HAR': 'HAQER0382', 'sig': '↓ protective',
     'rsID': 'rs925059852',  'vep': '.'},
    {'chrom': '2',  'pos': 207045,    'ref': 'G',   'alt': 'A',
     'OR': 0.702, 'P': 2.43e-10, 'HAR': 'HAQER0382', 'sig': '↓ protective',
     'rsID': 'rs868111032',  'vep': '.'},
    {'chrom': '2',  'pos': 207053,    'ref': 'G',   'alt': 'A',
     'OR': 0.653, 'P': 5.87e-11, 'HAR': 'HAQER0382', 'sig': '↓ protective',
     'rsID': 'rs1478551279', 'vep': '.'},
    {'chrom': '4',  'pos': 189810821, 'ref': 'G',   'alt': 'A',
     'OR': 0.442, 'P': 2.68e-10, 'HAR': 'HAQER0441', 'sig': '↓ protective',
     'rsID': 'rs13146781',   'vep': 'enhancer/FRG1-DT'},
    {'chrom': '7',  'pos': 158099346, 'ref': 'A',   'alt': 'C',
     'OR': 0.428, 'P': 2.00e-6,  'HAR': 'HAQER1329', 'sig': '↓ protective',
     'rsID': 'rs369458880',  'vep': 'PTPRN2'},
]

print(f'{len(VARIANTS_T1)} Tier 1 variants loaded')
for v in VARIANTS_T1:
    print(f"  {v['HAR']} chr{v['chrom']}:{v['pos']} {v['ref']}>{v['alt']}  OR={v['OR']}  {v['sig']}")

In [ ]:
# Load Enformer: extract the concrete function
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

print("Loading Enformer...")
_hub_module    = hub.load("https://tfhub.dev/deepmind/enformer/1")
_enformer_obj  = _hub_module.model

# Extract the concrete function (works around a None bug in modern TF)
_input_spec  = tf.TensorSpec(shape=[None, 393216, 4], dtype=tf.float32)
_concrete_fn = _enformer_obj.predict_on_batch.get_concrete_function(_input_spec)

# Check which keys the function produces
print("Output keys:", list(_concrete_fn.structured_outputs.keys()))
print("Enformer ready.")

In [ ]:
# Download track metadata
import io, requests, pandas as pd

# Basenji/Enformer (DeepMind) original repo URL
TRACKS_URL = ('https://raw.githubusercontent.com/calico/basenji/master'
              '/manuscripts/cross2020/targets_human.txt')

resp = requests.get(TRACKS_URL, timeout=30)
resp.raise_for_status()

tracks_df = pd.read_csv(io.StringIO(resp.text), sep='\t')
tracks_df = tracks_df.reset_index(drop=True)

# Normalize the description column name
desc_col = [c for c in tracks_df.columns if 'description' in c.lower() or 'identifier' in c.lower()]
print('Columns:', tracks_df.columns.tolist())
print(f'Tracks loaded: {len(tracks_df)}')
print(tracks_df.head(3))

In [ ]:
# Helper functions
import numpy as np, requests, time

SEQ_LENGTH = 393216
BASES_MAP  = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

def one_hot_encode(seq):
    enc = np.zeros((len(seq), 4), dtype=np.float32)
    for i, b in enumerate(seq.upper()):
        idx = BASES_MAP.get(b, -1)
        if idx >= 0: enc[i, idx] = 1.0
    return enc

def fetch_sequence(chrom, center_pos_1based, length=SEQ_LENGTH):
    half  = length // 2
    start = max(0, center_pos_1based - 1 - half)
    end   = start + length
    url   = (f'https://api.genome.ucsc.edu/getData/sequence'
             f'?genome=hg38;chrom=chr{chrom};start={start};end={end}')
    resp  = requests.get(url, timeout=120)
    resp.raise_for_status()
    seq = resp.json()['dna'].upper()
    if len(seq) < length:
        seq = seq + 'N' * (length - len(seq))
    return seq, start

def make_alt_sequence(ref_seq, var_idx, ref_allele, alt_allele, length=SEQ_LENGTH):
    alt = ref_seq[:var_idx] + alt_allele + ref_seq[var_idx + len(ref_allele):]
    if   len(alt) < length: alt = alt + 'N' * (length - len(alt))
    elif len(alt) > length:
        trim = (len(alt) - length) // 2
        alt  = alt[trim: trim + length]
    return alt

def predict(seq_array):
    seq_tf = tf.constant(seq_array[np.newaxis], dtype=tf.float32)
    out    = _concrete_fn(seq_tf)               # concrete function (not predict_on_batch)
    key    = 'human' if 'human' in out else list(out.keys())[0]
    return out[key].numpy()[0]                  # (896, 5313)

def score_variant(v, n_center_bins=11):
    chrom, pos, ref, alt = v['chrom'], v['pos'], v['ref'], v['alt']
    label = f"{v['HAR']} chr{chrom}:{pos} {ref}>{alt}"
    print(f'\n{label}')

    seq, seq_start = fetch_sequence(chrom, pos)
    var_idx    = pos - 1 - seq_start
    actual_ref = seq[var_idx: var_idx + len(ref)]
    if actual_ref != ref:
        print(f'  REF mismatch: expected={ref}, got={actual_ref}')

    ref_enc = one_hot_encode(seq)
    alt_enc = one_hot_encode(make_alt_sequence(seq, var_idx, ref, alt))

    print('  Enformer REF...')
    pred_ref = predict(ref_enc)
    print('  Enformer ALT...')
    pred_alt = predict(alt_enc)

    center     = pred_ref.shape[0] // 2
    half_b     = n_center_bins // 2
    sl         = slice(center - half_b, center + half_b + 1)
    eps        = 1e-6
    delta_mean = np.log2((pred_alt[sl] + eps) / (pred_ref[sl] + eps)).mean(axis=0)

    print('  done')
    return {**v, 'label': label, 'pred_ref': pred_ref,
            'pred_alt': pred_alt, 'delta_mean': delta_mean}

print('Functions defined.')

In [ ]:
# Run Enformer on the 7 Tier 1 variants
results = []
for v in VARIANTS_T1:
    try:
        r = score_variant(v)
        results.append(r)
        time.sleep(1)
    except BaseException as e:
        import traceback
        print(f'  ERROR in {v["HAR"]} chr{v["chrom"]}:{v["pos"]}:')
        traceback.print_exc()

print(f'\n{len(results)}/{len(VARIANTS_T1)} variants processed')

In [ ]:
# Select tracks relevant to PD / neuronal regulation
KEYWORDS_INTEREST = [
    'H3K27ac',     # active enhancer
    'H3K4me1',     # poised enhancer
    'H3K4me3',     # promoter
    'DNASE',       # chromatin accessibility
    'ATAC',        # chromatin accessibility (alternative)
    'brain',       # brain tissue
    'neuro',       # neuronal tissue
    'CTCF',        # insulator
    'H3K9me3',     # heterochromatin / silencing
    'H3K36me3',    # active transcription
]

# Indices of tracks of interest
mask = tracks_df['description'].str.lower().str.contains(
    '|'.join([kw.lower() for kw in KEYWORDS_INTEREST])
)
tracks_interest = tracks_df[mask]
idx_interest    = tracks_interest.index.tolist()
print(f'Tracks of interest: {len(idx_interest)}')
tracks_interest.head(20)

In [ ]:
# Clean up invalid results
results = [r for r in results if r is not None and isinstance(r, dict) and 'delta_mean' in r]
print(f"{len(results)} valid results out of {len(VARIANTS_T1)} variants")
for r in results:
    print(f"  {r['label']}")

In [ ]:
# Summary table: top tracks most affected per variant
TOP_N = 15   # top N tracks to show

summary_rows = []
for r in results:
    delta = r['delta_mean']   # (5313,)
    # Filter to tracks of interest and take the top N by |delta|
    delta_interest = [(idx, delta[idx]) for idx in idx_interest]
    delta_interest.sort(key=lambda x: abs(x[1]), reverse=True)

    for rank, (idx, d) in enumerate(delta_interest[:TOP_N]):
        summary_rows.append({
            'HAR':          r['HAR'],
            'variant':      f"{r['ref']}>{r['alt']}",
            'pos':          r['pos'],
            'OR':           r['OR'],
            'sig':          r['sig'],
            'track_idx':    idx,
            'track_desc':   tracks_df.loc[idx, 'description'],
            'log2FC':       round(d, 4),
            'rank':         rank + 1,
        })

summary_df = pd.DataFrame(summary_rows)
print('Summary table:')
summary_df.head(30)

In [ ]:
# Visualization: delta scores per variant
fig, axes = plt.subplots(len(results), 1,
                         figsize=(14, 3 * len(results)),
                         constrained_layout=True)
if len(results) == 1:
    axes = [axes]

cmap_pos = '#C62828'   # red = signal increase
cmap_neg = '#1565C0'   # blue = signal decrease

for ax, r in zip(axes, results):
    delta = r['delta_mean']
    delta_top = [(idx, delta[idx]) for idx in idx_interest]
    delta_top.sort(key=lambda x: abs(x[1]), reverse=True)
    delta_top = delta_top[:TOP_N]

    labels = [tracks_df.loc[i, 'description'][:50] for i, _ in delta_top]
    values = [d for _, d in delta_top]
    colors = [cmap_pos if v > 0 else cmap_neg for v in values]

    bars = ax.barh(range(len(values)), values, color=colors, alpha=0.85)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('log₂FC (ALT/REF)', fontsize=8)
    title = (f"{r['HAR']}  chr{r['chrom']}:{r['pos']} {r['ref']}>{r['alt']}  "
             f"OR={r['OR']}  {r['sig']}  VEP={r.get('vep','.')}")
    ax.set_title(title, fontsize=8, fontweight='bold')
    ax.invert_yaxis()

plt.suptitle('Enformer - top affected regulatory tracks (log₂FC ALT vs REF)\n'
             'GP2 R12 HARs/HAQERs EUR WGS - Parkinson Disease',
             fontsize=10, fontweight='bold')
plt.savefig('enformer_delta_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: enformer_delta_scores.png')

In [ ]:
# Heatmap: all variants x H3K27ac and DNASE tracks
# Filter to H3K27ac and DNASE tracks for the paper's main figure
KEY_TRACKS = ['H3K27ac', 'DNASE', 'H3K4me1']

key_mask = tracks_df['description'].str.lower().str.contains(
    '|'.join([k.lower() for k in KEY_TRACKS])
)
key_idx = tracks_df[key_mask].index.tolist()[:50]  # top 50 tracks

# Build matrix: variants x tracks
matrix = np.array([r['delta_mean'][key_idx] for r in results])
row_labels  = [f"{r['HAR']} {r['ref']}>{r['alt']} ({r['sig']})"
               for r in results]
col_labels  = [tracks_df.loc[i, 'description'][:35] for i in key_idx]

vmax = np.percentile(np.abs(matrix), 95)

fig, ax = plt.subplots(figsize=(18, 4))
im = ax.imshow(matrix, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=8)
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=90, fontsize=6)
plt.colorbar(im, ax=ax, label='log₂FC (ALT/REF)', shrink=0.8)
ax.set_title('Enformer - log₂FC per variant x H3K27ac / DNASE / H3K4me1 tracks\n'
             'Red = ALT increases signal | Blue = ALT decreases signal',
             fontsize=9)
plt.tight_layout()
plt.savefig('enformer_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved: enformer_heatmap.png')

In [ ]:
# Save the summary table as TSV
out_path = 'enformer_scores_tier1_EUR_WGS.tsv'
summary_df.to_csv(out_path, sep='\t', index=False)
print(f'Table saved: {out_path}')

# Executive summary
print('\n' + '='*70)
print('ENFORMER SUMMARY - HAQER1191 Tier 1 variants (NCOR2 focus)')
print('='*70)
for r in results:
    if r['HAR'] == 'HAQER1191':
        delta = r['delta_mean']
        # Top H3K27ac track
        h3k27ac_idx = [i for i in idx_interest
                       if 'H3K27ac' in tracks_df.loc[i, 'description']]
        if h3k27ac_idx:
            best = max(h3k27ac_idx, key=lambda i: abs(delta[i]))
            print(f"\n{r['label']}")
            print(f"  OR={r['OR']}  {r['sig']}")
            print(f"  Top H3K27ac track: {tracks_df.loc[best,'description']}")
            print(f"  log2FC = {delta[best]:.4f} "
                  f"({'↑ more chromatin opening' if delta[best]>0 else '↓ less chromatin opening'})")